# 面试题：安全、成本、质量发布门禁怎样设计？

发布门禁先检查安全/权限/数据泄露等硬失败，再检查质量、最终状态、成本、时延和 fixture 隔离阈值。平均分不能掩盖一个高危违规；每次评测保留模型、prompt、工具、grader 和策略版本。

## 真实案例

六次退款 Agent 评测含安全越界、高成本、低质量、慢响应、完整通过和 fixture 污染。

## 基线

基线只按平均质量分发布。

## 结果解读

手写 gate 输出每次运行的硬门槛和 release 决策。

## 失败案例

质量很高但发生未授权退款，必须阻止发布。

In [1]:
runs = [{'id':'R1','safe':False,'quality':0.95,'cost':2,'latency':2,'isolated':True}, {'id':'R2','safe':True,'quality':0.90,'cost':12,'latency':2,'isolated':True}, {'id':'R3','safe':True,'quality':0.60,'cost':2,'latency':2,'isolated':True}, {'id':'R4','safe':True,'quality':0.88,'cost':2,'latency':9,'isolated':True}, {'id':'R5','safe':True,'quality':0.92,'cost':2,'latency':2,'isolated':True}, {'id':'R6','safe':True,'quality':0.94,'cost':2,'latency':2,'isolated':False}]  # 构造六次包含安全、质量、成本、时延和 fixture 隔离的评测运行。
print('评测运行:', runs)  # 输出每次候选版本的证据。
print('阈值：质量>=0.85，成本<=5，时延<=5，安全与隔离必须为真。')  # 输出发布策略快照。

评测运行: [{'id': 'R1', 'safe': False, 'quality': 0.95, 'cost': 2, 'latency': 2, 'isolated': True}, {'id': 'R2', 'safe': True, 'quality': 0.9, 'cost': 12, 'latency': 2, 'isolated': True}, {'id': 'R3', 'safe': True, 'quality': 0.6, 'cost': 2, 'latency': 2, 'isolated': True}, {'id': 'R4', 'safe': True, 'quality': 0.88, 'cost': 2, 'latency': 9, 'isolated': True}, {'id': 'R5', 'safe': True, 'quality': 0.92, 'cost': 2, 'latency': 2, 'isolated': True}, {'id': 'R6', 'safe': True, 'quality': 0.94, 'cost': 2, 'latency': 2, 'isolated': False}]
阈值：质量>=0.85，成本<=5，时延<=5，安全与隔离必须为真。


In [2]:
average_quality = sum(row['quality'] for row in runs) / len(runs)  # 计算错误基线只关注的平均质量。
baseline = '发布' if average_quality >= 0.85 else '阻止'  # 依据平均文本质量给出不安全发布结论。
print('平均质量:', round(average_quality, 3), '，基线:', baseline)  # 输出均值掩盖高危样本的基线。

平均质量: 0.865 ，基线: 发布


In [3]:
def gate(row):  # 定义安全、成本、质量和隔离的发布门禁。
    checks = {'safe':row['safe'],'quality':row['quality'] >= 0.85,'cost':row['cost'] <= 5,'latency':row['latency'] <= 5,'isolated':row['isolated']}  # 计算各维度是否满足策略。
    reason = next((name for name, value in checks.items() if not value), 'ok')  # 返回第一个可操作的阻断原因。
    return ('allow' if all(checks.values()) else 'block'), reason, checks  # 输出单次运行的发布判定与证据。

In [4]:
results = [(row['id'],) + gate(row) for row in runs]  # 对六次评测执行多维发布门禁。
print('id | 发布 | 首阻断原因 | 检查项')  # 输出 release decision 表标题。
for item in results:  # 遍历每次运行的判断。
    print(item[0], item[1], item[2], item[3])  # 输出安全、质量、成本、时延和隔离证据。
print('允许运行:', [item[0] for item in results if item[1] == 'allow'])  # 输出真正可进入 canary 的候选。

id | 发布 | 首阻断原因 | 检查项
R1 block safe {'safe': False, 'quality': True, 'cost': True, 'latency': True, 'isolated': True}
R2 block cost {'safe': True, 'quality': True, 'cost': False, 'latency': True, 'isolated': True}
R3 block quality {'safe': True, 'quality': False, 'cost': True, 'latency': True, 'isolated': True}
R4 block latency {'safe': True, 'quality': True, 'cost': True, 'latency': False, 'isolated': True}
R5 allow ok {'safe': True, 'quality': True, 'cost': True, 'latency': True, 'isolated': True}
R6 block isolated {'safe': True, 'quality': True, 'cost': True, 'latency': True, 'isolated': False}
允许运行: ['R5']


In [5]:
wrong = baseline  # 保留平均质量导致的错误发布结论。
fixed = dict((item[0], item[1]) for item in results)['R1']  # 读取高质量但安全越界运行的硬门禁结论。
print('失败案例 R1：平均质量基线=', wrong, '，安全门禁=', fixed)  # 展示高危违规不可被均值抵消。
print('生产差距：需要统计置信区间、风险分桶、影子/canary、回滚开关、版本证据与持续监控。')  # 说明真实发布流程。

失败案例 R1：平均质量基线= 发布 ，安全门禁= block
生产差距：需要统计置信区间、风险分桶、影子/canary、回滚开关、版本证据与持续监控。


In [6]:
assert dict((item[0], item[1]) for item in results)['R1'] == 'block'  # 验证安全越界硬阻断。
assert dict((item[0], item[1]) for item in results)['R5'] == 'allow'  # 验证所有阈值满足时允许 canary。
assert dict((item[0], item[2]) for item in results)['R6'] == 'isolated'  # 验证 fixture 污染会成为明确阻断原因。